In [8]:
## main_pipeline.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/MTM_Risk_project/data/raw.csv'
SCORED_PATH = Path('/content/drive/MyDrive/MTM_Risk_project/data/scored_data.csv')

# Load the raw data into a DataFrame
df = pd.read_csv(path)

# -----------------------------
# Basic cleaning
# -----------------------------
# Fill missing values safely
if "adherence_score" in df.columns:
    df["adherence_score"] = df["adherence_score"].fillna(df["adherence_score"].median())
else:
    raise ValueError("Missing required column: adherence_score")

for col in ["med_count", "comorbidity_count", "last_hospital"]:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)
    else:
        raise ValueError(f"Missing required column: {col}")

if "a1c" in df.columns:
    df["a1c"] = df["a1c"].fillna(df["a1c"].median())

# Convert adherence from 0–100 to 0–1 if needed
if df["adherence_score"].max() > 1.5:
    df["adherence_score"] = df["adherence_score"] / 100.0

# Clip adherence to valid range
df["adherence_score"] = df["adherence_score"].clip(0, 1)

# -----------------------------
# Derived clinical flags
# -----------------------------
df["polypharmacy"] = (df["med_count"] >= 5).astype(int)
df["poor_adherence"] = (df["adherence_score"] < 0.6).astype(int)
df["high_comorbidity"] = (df["comorbidity_count"] >= 3).astype(int)

print("\nDerived flags:")
print(df[["med_count", "adherence_score", "comorbidity_count", "last_hospital",
          "polypharmacy", "poor_adherence", "high_comorbidity"]].head())

# -----------------------------
# Normalize features
# -----------------------------
max_med = max(df["med_count"].max(), 1)
max_comorb = max(df["comorbidity_count"].max(), 1)

df["med_norm"] = df["med_count"] / max_med
df["comorb_norm"] = df["comorbidity_count"] / max_comorb
df["adherence_risk"] = 1 - df["adherence_score"]
df["hospital_norm"] = df["last_hospital"].clip(0, 1)

# -----------------------------
# Risk scoring models
# -----------------------------
# Model A: baseline
# med 0.4 + adherence 0.3 + hospital 0.2 + comorbidity 0.1
df["risk_score_A"] = (
    df["med_norm"] * 0.40 +
    df["adherence_risk"] * 0.30 +
    df["hospital_norm"] * 0.20 +
    df["comorb_norm"] * 0.10
)

# Model B: hospitalization-focused
# med 0.3 + adherence 0.2 + hospital 0.4 + comorbidity 0.1
df["risk_score_B"] = (
    df["med_norm"] * 0.30 +
    df["adherence_risk"] * 0.20 +
    df["hospital_norm"] * 0.40 +
    df["comorb_norm"] * 0.10
)

# Model C: adherence and safety-focused (selected)
# med 0.25 + adherence 0.4 + hospital 0.2 + comorbidity 0.15
df["risk_score_C"] = (
    df["med_norm"] * 0.25 +
    df["adherence_risk"] * 0.40 +
    df["hospital_norm"] * 0.20 +
    df["comorb_norm"] * 0.15
)

# Round for presentation
for col in ["risk_score_A", "risk_score_B", "risk_score_C"]:
    df[col] = df[col].round(3)

print("\nRisk score preview:")
print(df[["patient_id", "risk_score_A", "risk_score_B", "risk_score_C"]].head())

# -----------------------------
# Risk tiering
# -----------------------------
final_score = df["risk_score_C"]
low_cutoff = final_score.quantile(0.50)
high_cutoff = final_score.quantile(0.80)


def assign_risk_tier(score):
    if score >= high_cutoff:
        return "High"
    elif score >= low_cutoff:
        return "Medium"
    else:
        return "Low"


df["actual_risk"] = final_score.apply(assign_risk_tier)
print("\nLow cutoff:", round(low_cutoff, 3))
print("High cutoff:", round(high_cutoff, 3))
print(df["actual_risk"].value_counts())

# -----------------------------
# Safety rules
# -----------------------------
# Safety logic overrides under-called high-risk cases

def apply_safety_rules(row):
    if row["adherence_score"] < 0.5 and row["last_hospital"] == 1:
        return "High"

    if row["med_count"] >= 8 and row["comorbidity_count"] >= 3:
        return "High"

    if row["adherence_score"] < 0.4:
        return "High"

    return row["actual_risk"]


df["final_risk"] = df.apply(apply_safety_rules, axis=1)

# -----------------------------
# Explanation engine
# -----------------------------

def explain_patient(row):
    drivers = []

    if row["med_count"] >= 5:
        drivers.append("polypharmacy")
    if row["adherence_score"] < 0.6:
        drivers.append("poor adherence")
    if row["last_hospital"] == 1:
        drivers.append("recent hospitalization")
    if row["comorbidity_count"] >= 3:
        drivers.append("multiple comorbidities")

    if len(drivers) == 0:
        return "No major risk drivers detected."

    return "Main drivers: " + ", ".join(drivers[:3])


df["explanation"] = df.apply(explain_patient, axis=1)

# -----------------------------
# Recommendation engine
# -----------------------------

def recommend_action(row):
    if row["final_risk"] == "High":
        if row["polypharmacy"] == 1 and row["poor_adherence"] == 1:
            return "Immediate pharmacist medication review and adherence counseling"
        elif row["last_hospital"] == 1:
            return "Post-discharge MTM follow-up and medication reconciliation"
        else:
            return "High-priority MTM review"

    elif row["final_risk"] == "Medium":
        if row["comorbidity_count"] >= 3:
            return "Standard MTM outreach with chronic disease focus"
        else:
            return "Monitor and reassess soon"

    else:
        return "Routine follow-up and patient education"


df["recommendation"] = df.apply(recommend_action, axis=1)

# -----------------------------
# Save output
# -----------------------------
SCORED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(SCORED_PATH, index=False)

print("\nSaved final scored dataset to:", SCORED_PATH)
print(df[["patient_id", "final_risk", "risk_score_C", "explanation", "recommendation"]].head(10))

# ── ML UPGRADE ───────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Use actual_risk (pre-safety-rules) as proxy label to avoid leakage
df["target_high_risk"] = (df["actual_risk"] == "High").astype(int)

features = ["med_count", "adherence_score", "comorbidity_count", "last_hospital"]
X = df[features]
y = df["target_high_risk"]

# Scale features — important for logistic regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Train model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nML Model Performance:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

# ── K-FOLD CROSS-VALIDATION ──────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Use the same features and target as before
# X_scaled and y are already defined above

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression cross-validation
lr_cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_scaled, y,
    cv=kf,
    scoring="roc_auc"
)

# Random Forest cross-validation
rf_cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_scaled, y,
    cv=kf,
    scoring="roc_auc"
)

print("\n── K-Fold Cross-Validation Results (5 folds) ───────────────────")
print(f"\nLogistic Regression:")
print(f"  Fold scores : {[float(round(s, 4)) for s in lr_cv_scores]}")
print(f"  Fold scores : {[float(round(s, 4)) for s in rf_cv_scores]}")
print(f"  Mean ROC-AUC: {lr_cv_scores.mean():.4f}")
print(f"  Std dev     : {lr_cv_scores.std():.4f}  (lower = more stable)")

print(f"\nRandom Forest:")
print(f"  Fold scores : {[round(float(s), 4) for s in rf_cv_scores]}")
print(f"  Mean ROC-AUC: {rf_cv_scores.mean():.4f}")
print(f"  Std dev     : {rf_cv_scores.std():.4f}  (lower = more stable)")

print(f"\nInterpretation:")
print(f"  LR  single split: 0.998  vs  CV mean: {lr_cv_scores.mean():.4f}")
print(f"  RF  single split: 0.995  vs  CV mean: {rf_cv_scores.mean():.4f}")

# Save CV results to df for dashboard use
df["lr_cv_mean"] = round(lr_cv_scores.mean(), 4)
df["lr_cv_std"]  = round(lr_cv_scores.std(),  4)
df["rf_cv_mean"] = round(rf_cv_scores.mean(), 4)
df["rf_cv_std"]  = round(rf_cv_scores.std(),  4)

# Feature importance (coefficients)
coef_df = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0].round(3)
}).sort_values("coefficient", ascending=False)
print("\nFeature Importance (Logistic Regression Coefficients):")
print(coef_df)

# Save ML probability for ALL patients
df["ml_risk_prob"] = model.predict_proba(X_scaled)[:, 1].round(3)

# Use percentile-based tiers to match rule-based approach
low_ml = df["ml_risk_prob"].quantile(0.50)
high_ml = df["ml_risk_prob"].quantile(0.80)

df["ml_risk_tier"] = df["ml_risk_prob"].apply(
    lambda p: "High" if p >= high_ml else ("Medium" if p >= low_ml else "Low")
)

# Comparison column — do both systems agree?
df["tier_match"] = (df["final_risk"] == df["ml_risk_tier"])

print("\nTier agreement (rule-based vs ML):")
print(df["tier_match"].value_counts())



# ── RANDOM FOREST MODEL ──────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

# Same features and target as logistic regression
# X_scaled and y are already defined above — no need to redefine

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("\nRandom Forest Performance:")
print(classification_report(y_test, rf_pred))
print("RF ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 3))

# Feature importance (different from coefficients — this is based on how
# much each feature reduces uncertainty across all 100 trees)
rf_importance = pd.DataFrame({
    "feature": features,
    "importance": rf_model.feature_importances_.round(3)
}).sort_values("importance", ascending=False)

print("\nRandom Forest Feature Importance:")
print(rf_importance)

# Save RF probability for all patients
df["rf_risk_prob"] = rf_model.predict_proba(X_scaled)[:, 1].round(3)

# RF needs percentile-based thresholds because its probabilities
# are heavily skewed toward zero
low_rf  = df["rf_risk_prob"].quantile(0.50)
high_rf = df["rf_risk_prob"].quantile(0.75)  # 75th not 80th — adjusted for RF distribution

df["rf_risk_tier"] = df["rf_risk_prob"].apply(
    lambda p: "High" if p >= high_rf else ("Medium" if p >= low_rf else "Low")
)

# But handle the edge case where low_rf = 0 (too many zeros)
# In that case use a small positive threshold
if low_rf == 0:
    low_rf = 0.001

df["rf_risk_tier"] = df["rf_risk_prob"].apply(
    lambda p: "High" if p >= high_rf else ("Medium" if p >= low_rf else "Low")
)

df["rf_tier_match"] = (df["final_risk"] == df["rf_risk_tier"])
print(f"RF tier agreement (final): {df['rf_tier_match'].sum()}/800")
print(df["rf_risk_tier"].value_counts())

# Model comparison summary
print("\n── Model Comparison ─────────────────────────────────────")
print(f"Logistic Regression  ROC-AUC : {round(roc_auc_score(y_test, y_prob), 3)}")
print(f"Random Forest        ROC-AUC : {round(roc_auc_score(y_test, rf_prob), 3)}")
print(f"\nLR  tier agreement with rule-based : {df['tier_match'].sum()}/800")
rf_tier_match = (df["final_risk"] == df["rf_risk_tier"])
print(f"RF  tier agreement with rule-based : {rf_tier_match.sum()}/800")

# ── ENSEMBLE MODEL (LR + RF Combined) ───────────────────────────────────────

# Average probability from both models
df["ensemble_prob"] = ((df["ml_risk_prob"] + df["rf_risk_prob"]) / 2).round(3)

# Assign ensemble tiers using percentiles
low_ens  = df["ensemble_prob"].quantile(0.50)
high_ens = df["ensemble_prob"].quantile(0.80)

df["ensemble_tier"] = df["ensemble_prob"].apply(
    lambda p: "High" if p >= high_ens else ("Medium" if p >= low_ens else "Low")
)

# Agreement with rule-based
df["ensemble_match"] = (df["final_risk"] == df["ensemble_tier"])

# Confidence flag — all three models agree
df["all_agree"] = (
    (df["final_risk"] == df["ml_risk_tier"]) &
    (df["final_risk"] == df["rf_risk_tier"]) &
    (df["final_risk"] == df["ensemble_tier"])
)

# Full model comparison summary
print("\n── Final Model Comparison ───────────────────────────────────")
print(f"Rule-based only         : baseline")
print(f"Logistic Regression     ROC-AUC : {round(roc_auc_score(y_test, y_prob), 3)}  |  Tier agreement : {df['tier_match'].sum()}/800")
print(f"Random Forest           ROC-AUC : {round(roc_auc_score(y_test, rf_prob), 3)}  |  Tier agreement : {df['rf_tier_match'].sum()}/800")
print(f"Ensemble (LR + RF)      -        |  Tier agreement : {df['ensemble_match'].sum()}/800")
print(f"\nPatients where ALL models agree : {df['all_agree'].sum()}/800")
print(f"Patients where models disagree  : {(~df['all_agree']).sum()}/800")
print("\nEnsemble tier distribution:")
print(df["ensemble_tier"].value_counts())

# Save final output
df.to_csv(SCORED_PATH, index=False)
print("\nSaved to:", SCORED_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Derived flags:
   med_count  adherence_score  comorbidity_count  last_hospital  polypharmacy  \
0          4             0.52                  2              0             0   
1          9             0.79                  3              1             1   
2          1             0.83                  2              1             0   
3          7             0.93                  3              0             1   
4          4             0.76                  5              0             0   

   poor_adherence  high_comorbidity  
0               1                 0  
1               0                 1  
2               0                 0  
3               0                 1  
4               0                 1  

Risk score preview:
  patient_id  risk_score_A  risk_score_B  risk_score_C
0      P0001         0.279         0.205         0.302
1      P0

In [ ]:
print("RF probability distribution:")
print(df["rf_risk_prob"].describe().round(3))
print("\nValue counts by RF tier:")
print(df["rf_risk_tier"].value_counts())
print("\nValue counts by rule-based final_risk:")
print(df["final_risk"].value_counts())

RF probability distribution:
count    800.000
mean       0.202
std        0.375
min        0.000
25%        0.000
50%        0.000
75%        0.080
max        1.000
Name: rf_risk_prob, dtype: float64

Value counts by RF tier:
rf_risk_tier
Low       625
High      161
Medium     14
Name: count, dtype: int64

Value counts by rule-based final_risk:
final_risk
Low       363
High      263
Medium    174
Name: count, dtype: int64
